# Experiment 13 — Random Forest HPT on the best target-feature set

This experiment uses Exp 9's best 18 features: compact features, global 24-month target lag, town–flat-type historical mean, and nearest MRT / attraction / monument names. It tunes a random forest with chronological CV, then evaluates once on the final untouched validation period.

**Executed result:** the tuned forest scored **RMSE 479.319**, **MAE 350.676**, and **R² 0.4677**. The best sampled configuration used 355 trees, depth 28, `max_features≈0.518`, minimum leaf size 6, and minimum split size 7. Tuning improved the random-forest family but did not close the gap to boosted trees.

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents]
project_root = next((root for root in roots if (root / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets

TARGET = 'MONTHLY_RENT'
GROUP_COLUMNS = ['TOWN', 'FLAT_TYPE']
LAG_MONTHS = 24
N_SPLITS = int(os.environ.get('RF_HPT_SPLITS', '3'))
N_ITERATIONS = int(os.environ.get('RF_HPT_ITERATIONS', '12'))
RANDOM_STATE = 42

compact_features = ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km']
BEST_FEATURES = compact_features + ['monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name', 'nearest_tourist_attraction_name', 'nearest_monument_name']


In [2]:
class TargetFeatureTransformer:
    def fit(self, history):
        self.global_mean_ = history[TARGET].mean()
        self.group_mean_ = history.groupby(GROUP_COLUMNS)[TARGET].mean()
        self.monthly_mean_ = history.groupby('period')[TARGET].mean()
        return self

    def transform(self, frame):
        result = frame[GROUP_COLUMNS + ['period']].copy()
        result['monthly_target_lag'] = (result['period'] - LAG_MONTHS).map(self.monthly_mean_).fillna(self.global_mean_)
        group_mean = pd.Series(result.set_index(GROUP_COLUMNS).index.map(self.group_mean_), index=result.index)
        result['town_flat_type_target_mean'] = group_mean.fillna(self.global_mean_)
        return result[['monthly_target_lag', 'town_flat_type_target_mean']]

    def fit_transform_train(self, history):
        self.fit(history)
        grouped = history.groupby(GROUP_COLUMNS + ['period'], as_index=False)[TARGET].agg(['sum', 'count']).reset_index().sort_values(GROUP_COLUMNS + ['period'])
        grouped['prior_sum'] = grouped.groupby(GROUP_COLUMNS)['sum'].cumsum() - grouped['sum']
        grouped['prior_count'] = grouped.groupby(GROUP_COLUMNS)['count'].cumsum() - grouped['count']
        grouped['town_flat_type_target_mean'] = grouped['prior_sum'] / grouped['prior_count'].replace(0, np.nan)
        global_monthly = history.groupby('period')[TARGET].agg(['sum', 'count']).sort_index()
        global_prior = (global_monthly['sum'].cumsum() - global_monthly['sum']) / (global_monthly['count'].cumsum() - global_monthly['count']).replace(0, np.nan)
        result = history[GROUP_COLUMNS + ['period']].merge(grouped[GROUP_COLUMNS + ['period', 'town_flat_type_target_mean']], on=GROUP_COLUMNS + ['period'], how='left')
        result['town_flat_type_target_mean'] = result['town_flat_type_target_mean'].fillna(result['period'].map(global_prior))
        result['monthly_target_lag'] = (result['period'] - LAG_MONTHS).map(self.monthly_mean_).fillna(result['period'].map(global_prior))
        return result[['monthly_target_lag', 'town_flat_type_target_mean']]

def nearest_mrt_name(base):
    stations = datasets['mrt'][['NAME', 'LATITUDE', 'LONGITUDE']].dropna()
    _, index = cKDTree(stations[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96]).query(base[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96])
    return pd.Series(stations['NAME'].astype(object).to_numpy()[index], name='nearest_mrt_name')

def nearest_geojson_name(base, filename, field, name):
    with (project_root / 'data' / 'possible_additional_features' / filename).open() as handle:
        data = json.load(handle)
    points = pd.DataFrame([
        {'longitude': item['geometry']['coordinates'][0], 'latitude': item['geometry']['coordinates'][1], name: item['properties'][field]}
        for item in data['features'] if item.get('geometry', {}).get('type') == 'Point'
    ]).dropna()
    _, index = cKDTree(points[['longitude', 'latitude']].to_numpy() * [110.96, 111.0]).query(base[['LONGITUDE', 'LATITUDE']].to_numpy() * [110.96, 111.0])
    return pd.Series(points[name].astype(object).to_numpy()[index], name=name)

def build_best_features(history_raw, future_raw):
    processor = RentalPreprocessor(datasets)
    history_base = processor.fit_transform(history_raw.drop(columns=['period', TARGET], errors='ignore'))
    future_base = processor.transform(future_raw.drop(columns=['period', TARGET], errors='ignore'))
    history = history_base[GROUP_COLUMNS].reset_index(drop=True).assign(period=history_raw['period'].to_numpy(), **{TARGET: history_raw[TARGET].to_numpy()})
    future = future_base[GROUP_COLUMNS].reset_index(drop=True).assign(period=future_raw['period'].to_numpy())
    transformer = TargetFeatureTransformer()
    history_target = transformer.fit_transform_train(history)
    future_target = transformer.transform(future)
    def combine(base, target):
        return pd.concat([
            base[compact_features].reset_index(drop=True), target.reset_index(drop=True),
            nearest_mrt_name(base).reset_index(drop=True),
            nearest_geojson_name(base, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name').reset_index(drop=True),
            nearest_geojson_name(base, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name').reset_index(drop=True),
        ], axis=1)[BEST_FEATURES]
    return combine(history_base, history_target), combine(future_base, future_target)


In [3]:
_, datasets = load_datasets(project_root / 'data')
raw = datasets['train'].copy()
raw['period'] = pd.to_datetime(raw['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
periods = np.sort(raw['period'].unique())
cutoff = periods[int(len(periods) * 0.80)]
raw_fit = raw.loc[raw['period'] < cutoff].copy()
raw_valid = raw.loc[raw['period'] >= cutoff].copy()
fit_x, valid_x = build_best_features(raw_fit, raw_valid)
fit_y = raw_fit[TARGET].astype(float).reset_index(drop=True)
valid_y = raw_valid[TARGET].astype(float).reset_index(drop=True)
fit_period = raw_fit['period'].reset_index(drop=True)
print(f'Outer cutoff={cutoff}; fit={fit_x.shape}; validation={valid_x.shape}; HPT iterations={N_ITERATIONS}')

categorical_columns = fit_x.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
numeric_columns = [column for column in BEST_FEATURES if column not in categorical_columns]
encoder = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median'))]), numeric_columns),
    ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_columns),
])
pipeline = Pipeline([
    ('encode', encoder),
    ('forest', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

# Produce row indices from month-based chronological splits, so a later month
# is never evaluated by a forest trained on that month or later months.
fit_periods = np.sort(fit_period.unique())
cv_splits = []
for train_month_index, validation_month_index in TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods):
    train_months = fit_periods[train_month_index]
    validation_months = fit_periods[validation_month_index]
    cv_splits.append((np.flatnonzero(fit_period.isin(train_months)), np.flatnonzero(fit_period.isin(validation_months))))
print([(len(train_index), len(validation_index)) for train_index, validation_index in cv_splits])


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


In [4]:
parameter_space = {
    'forest__n_estimators': randint(250, 651),
    'forest__max_depth': [None, 14, 20, 28, 36],
    'forest__min_samples_split': randint(2, 16),
    'forest__min_samples_leaf': randint(1, 11),
    'forest__max_features': uniform(0.30, 0.70),
    'forest__bootstrap': [True],
}
search = RandomizedSearchCV(
    estimator=pipeline, param_distributions=parameter_space, n_iter=N_ITERATIONS,
    scoring='neg_root_mean_squared_error', cv=cv_splits, refit=True,
    random_state=RANDOM_STATE, n_jobs=1, verbose=2, return_train_score=True,
)
search.fit(fit_x, fit_y)
print('Best chronological-CV RMSE:', -search.best_score_)
print('Best parameters:', search.best_params_)

validation_prediction = np.clip(search.best_estimator_.predict(valid_x), a_min=0, a_max=None)
validation_metrics = pd.DataFrame([{
    'model': 'Random Forest (chronological HPT)',
    'RMSE': mean_squared_error(valid_y, validation_prediction) ** 0.5,
    'MAE': mean_absolute_error(valid_y, validation_prediction),
    'R2': r2_score(valid_y, validation_prediction),
}])
display(validation_metrics)

cv_results = pd.DataFrame(search.cv_results_)
columns = ['rank_test_score', 'mean_test_score', 'std_test_score', 'mean_train_score', 'param_forest__n_estimators', 'param_forest__max_depth', 'param_forest__min_samples_split', 'param_forest__min_samples_leaf', 'param_forest__max_features']
display(cv_results[columns].sort_values('rank_test_score').head(10))


Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END forest__bootstrap=True, forest__max_depth=28, forest__max_features=0.9655000144869412, forest__min_samples_leaf=8, forest__min_samples_split=14, forest__n_estimators=270; total time=  13.2s
[CV] END forest__bootstrap=True, forest__max_depth=28, forest__max_features=0.9655000144869412, forest__min_samples_leaf=8, forest__min_samples_split=14, forest__n_estimators=270; total time=  43.4s
[CV] END forest__bootstrap=True, forest__max_depth=28, forest__max_features=0.9655000144869412, forest__min_samples_leaf=8, forest__min_samples_split=14, forest__n_estimators=270; total time= 1.6min
[CV] END forest__bootstrap=True, forest__max_depth=14, forest__max_features=0.40919616423534183, forest__min_samples_leaf=8, forest__min_samples_split=6, forest__n_estimators=349; total time=   5.8s
[CV] END forest__bootstrap=True, forest__max_depth=14, forest__max_features=0.40919616423534183, forest__min_samples_leaf=8, forest__min_sample

,model,RMSE,MAE,R2
0,Random Forest (chronological HPT),479.31934,350.675932,0.467733


,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_forest__n_estimators,param_forest__max_depth,param_forest__min_samples_split,param_forest__min_samples_leaf,param_forest__max_features
11,1,-587.559342,106.858816,-292.382050,355,28,7,6,0.518198
8,2,-587.726431,106.314870,-309.883033,595,28,10,10,0.694302
3,3,-588.189579,108.290344,-327.558231,563,14,2,5,0.427277
9,4,-588.924726,109.037686,-299.187737,513,36,13,7,0.461626
1,5,-588.977717,108.870417,-332.751151,349,14,6,8,0.409196
0,6,-589.035779,105.939504,-297.711462,270,28,14,8,0.965500
5,7,-589.620521,109.304276,-330.921640,304,14,13,3,0.332666
6,8,-589.951417,107.720214,-257.065245,556,28,6,3,0.714690
10,9,-589.953501,107.165018,-286.003880,609,20,3,4,0.936524
2,10,-590.380848,111.431574,-299.052532,543,20,9,2,0.314409


In [5]:
best_encoder = search.best_estimator_.named_steps['encode']
best_forest = search.best_estimator_.named_steps['forest']
importance = pd.DataFrame({'encoded_feature': best_encoder.get_feature_names_out(), 'importance': best_forest.feature_importances_})

def source_feature(name):
    if name.startswith('numeric__'):
        return name.removeprefix('numeric__')
    return next((column for column in categorical_columns if name.startswith(f'categorical__{column}_')), 'other')

importance['source_feature'] = importance['encoded_feature'].map(source_feature)
display(importance.groupby('source_feature', as_index=False)['importance'].sum().sort_values('importance', ascending=False))
display(importance.nlargest(25, 'importance'))


,source_feature,importance
17,town_flat_type_target_mean,0.398883
8,approval_date_num,0.186287
10,coe_quota_premium_mean,0.098635
3,FLOOR_AREA_SQM,0.057270
13,mrt_dist_nearest_km,0.042952
11,mall_dist_nearest_km,0.038377
7,YEAR_COMPLETED,0.036665
4,MAX_FLOOR,0.031205
2,FLAT_TYPE,0.022636
12,monthly_target_lag,0.017467


,encoded_feature,importance,source_feature
9,numeric__town_flat_type_target_mean,0.398883,town_flat_type_target_mean
1,numeric__approval_date_num,0.186287,approval_date_num
3,numeric__coe_quota_premium_mean,0.098635,coe_quota_premium_mean
0,numeric__FLOOR_AREA_SQM,0.057270,FLOOR_AREA_SQM
6,numeric__mrt_dist_nearest_km,0.042952,mrt_dist_nearest_km
7,numeric__mall_dist_nearest_km,0.038377,mall_dist_nearest_km
4,numeric__YEAR_COMPLETED,0.036665,YEAR_COMPLETED
5,numeric__MAX_FLOOR,0.031205,MAX_FLOOR
8,numeric__monthly_target_lag,0.017467,monthly_target_lag
3311,categorical__FLAT_TYPE_3 room,0.015295,FLAT_TYPE
